In [2]:
import pandas as pd
import numpy as np

def kaufman_efficiency_ratio(close_series, period=10):
    # Variation nette sur la période (entre maintenant et il y a N périodes)
    net_change = np.abs(close_series - close_series.shift(period))
    
    # Somme des variations absolues bar à bar sur la période
    change = np.abs(close_series - close_series.shift(1))
    volatility = change.rolling(window=period).sum()
    
    # Éviter la division par zéro
    er = np.where(volatility == 0, 0, net_change / volatility)
    
    return pd.Series(er, index=close_series.index)

# Exemple d'utilisation :
# df['KER'] = kaufman_efficiency_ratio(df['Close'], period=10)

In [3]:
import numpy as np
import pandas as pd


def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = "NEUTRAL"

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [13]:

from backtesting import Strategy

global_tp_multiple = 0
global_stop_trade_hour = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple = 3
    start_trade_hour = 5
    stop_trade_hour = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.order_placed_hour:
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.order_placed_hour:
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price

        return int(min(share_by_risk, share_by_leverage))

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple 
        tp_price = planned_entry_price + per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.order_placed_hour)
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple 
        tp_price = planned_entry_price - per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.order_placed_hour)
            self.current_day_order_placed = True
            
    def use_rsi_strategy(self, candle_datetime):
            if self.last_low is None:
                self.last_low = self.data.Low[-1]
            else:
                if self.last_low > self.data.Low[-1]:
                    self.last_low = self.data.Low[-1]
    
            if self.last_high is None:
                self.last_high = self.data.High[-1]
            else:
                if self.last_high < self.data.High[-1]:
                    self.last_high = self.data.High[-1]
    
            c_long = self.data['RSI14'][-1] < 35
            c_short = self.data['RSI14'][-1] > 65
    
            c_market_structure = self.data['market_state'] == 'NEUTRAL'
            
            c_time_limit = candle_datetime.time().hour < self.stop_trade_hour and candle_datetime.time().hour > self.start_trade_hour
    
            if not self.current_day_order_placed:
                if c_long and c_time_limit and c_market_structure:
                    self.place_long_order(candle_datetime)
    
                if c_short and c_time_limit and c_market_structure:
                    self.place_short_order(candle_datetime)
                    
            else:
                for trade in self.trades:
                    if trade.tag == self.order_placed_hour:
                        self.order_executed = True
    
                if self.order_executed == False:
                    for order in self.orders:
                        if order.tag == self.order_placed_hour:
                            order.cancel()
                            self.current_day_order_placed = False
    
                            if c_time_limit and c_market_structure:
                                if order.is_long:
                                    self.place_long_order(candle_datetime)
                                else:
                                    self.place_short_order(candle_datetime)

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
        
        if self.data['ER'][-1] < self.er15_threshold:
            self.use_rsi_strategy(t)
                            
        # print("--------------------------------")
            



In [14]:
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

window = 2
global_tp_multiple = 4
global_stop_trade_hour = 16
global_start_trade_hour = 5
global_er15_threshold = 0.3

# start_date = '2023-3-20 00:00'
# end_date = '2023-7-7 00:00'
start_date = '2020-1-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"


df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    # er is previous day er
    if i == 0:
        er = row['ER']
    else :
        er = df_result.iloc[i-1]['ER']

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(row['market_state'])
        h1_tf_er15_list.append(er)

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list
indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-01-01 22:00...
End                       2026-09-16 01:00...
Duration                   2449 days 02:00:00
Exposure Time [%]                     9.88189
Equity Final [$]                     18998.14
Equity Peak [$]                      19847.47
Return [%]                            89.9814
Buy & Hold Return [%]                45.36014
Return (Ann.) [%]                     8.01727
Volatility (Ann.) [%]                12.51197
CAGR [%]                             10.04397
Sharpe Ratio                          0.64077
Sortino Ratio                          1.2814
Calmar Ratio                          0.55925
Alpha [%]                            90.50813
Beta                                 -0.01161
Max. Drawdown [%]                   -14.33563
Avg. Drawdown [%]                    -2.01258
Max. Drawdown Duration      320 days 08:00:00
Avg. Drawdown Duration       21 days 04:00:00
# Trades                                  229
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p2960', ...)

In [9]:
import time

from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

window = 2
global_tp_multiple = 3
global_stop_trade_hour = 16
global_start_trade_hour = 5
global_er15_threshold = 0.3

# start_date = '2021-3-1 00:00'
# end_date = '2023-3-30 00:00'
start_date = '2020-1-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"


df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    # er is previous day er
    if i == 0:
        er = row['ER']
    else :
        er = df_result.iloc[i-1]['ER']

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(row['market_state'])
        h1_tf_er15_list.append(er)

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list
indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

start_hour_list = [4,5,6]
stop_hour_list = [16,19,21]
tp_multiple_list = [1,2,3,4,5,6,7,8]

with open(f"./results/result_{pair}_{time.time()}.csv", "a") as f:
    f.write(f"StartHour,StopHour,TpMultiple,Return,ReturnYearly,MDrawdown,WinRate,NumTrades\n")

    for start_hour in start_hour_list:
        for stop_hour in stop_hour_list:
            for tp_multiple in tp_multiple_list:
                global_start_trade_hour = start_hour
                global_stop_trade_hour = stop_hour
                global_tp_multiple = tp_multiple

                bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                stats = bt.run()
        
                f.write(f"{global_start_trade_hour},{global_stop_trade_hour},{global_tp_multiple},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']}, {stats['# Trades']}\n")
        
                print(f"start_hour = {start_hour} & stop_trade_hour = {stop_hour} & tp_multiple = {tp_multiple} DONE")

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 4 & stop_trade_hour = 16 & tp_multiple = 1 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 4 & stop_trade_hour = 16 & tp_multiple = 2 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

...

start_hour = 6 & stop_trade_hour = 21 & tp_multiple = 6 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 6 & stop_trade_hour = 21 & tp_multiple = 7 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 6 & stop_trade_hour = 21 & tp_multiple = 8 DONE


In [ ]:
# rsi ranging stratey
from backtesting import Strategy

global_tp_multiple = 0
global_stop_trade_hour = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple = 3
    start_trade_hour = 5
    stop_trade_hour = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple 
        tp_price = planned_entry_price + per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple 
        tp_price = planned_entry_price - per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35
        c_short = self.data['RSI14'][-1] > 65

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_hour:
            self.reset_range(current_bar_date)

        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_market_structure = self.data['market_state'][-1] == 'NEUTRAL'
        c_er = self.data['ER'][-1] < self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_hour and candle_datetime.time().hour > self.start_trade_hour
        
        if c_market_structure and c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [47]:
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

window = 2
global_tp_multiple = 5
global_stop_trade_hour = 16
global_start_trade_hour = 5
global_er15_threshold = 0.3

# start_date = '2023-3-20 00:00'
# end_date = '2023-7-7 00:00'
start_date = '2020-1-1 00:00'
end_date = '2022-12-30 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"


df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
# Affichage des transitions d'états
# df_result["state_changed"] = df_result["market_state"] != df_result[
#     "market_state"
# ].shift(1)
# print(df_result[df_result["state_changed"]][["Close", "market_state"]])
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        h1_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list
indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-01-13 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-01-01 22:00...
End                       2022-12-30 00:00...
Duration                   1093 days 02:00:00
Exposure Time [%]                    15.58074
Equity Final [$]                     8546.163
Equity Peak [$]                      10555.88
Return [%]                          -14.53837
Buy & Hold Return [%]                11.00105
Return (Ann.) [%]                    -4.13716
Volatility (Ann.) [%]                13.53257
CAGR [%]                             -5.11412
Sharpe Ratio                         -0.30572
Sortino Ratio                        -0.45728
Calmar Ratio                         -0.20108
Alpha [%]                           -14.47489
Beta                                 -0.00577
Max. Drawdown [%]                   -20.57512
Avg. Drawdown [%]                     -5.6323
Max. Drawdown Duration      833 days 00:00:00
Avg. Drawdown Duration      155 days 21:00:00
# Trades                                  135
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p8072', ...)

In [51]:
# rsi rending strategy

from backtesting import Strategy

global_tp_multiple = 0
global_stop_trade_hour = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple = 5
    start_trade_hour = 5
    stop_trade_hour = 21
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple 
        tp_price = planned_entry_price + per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple 
        tp_price = planned_entry_price - per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35 and self.data['market_state'][-1] == 'BULLISH'
        c_short = self.data['RSI14'][-1] > 65 and self.data['market_state'][-1] == 'BEARISH'

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def use_rsi_entry_strategy_market_trending(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35 and self.data['market_state'][-1] == 'BULLISH'
        c_short = self.data['RSI14'][-1] > 65 and self.data['market_state'][-1] == 'BEARISH'

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_hour:
            self.reset_range(current_bar_date)

        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_market_structure = self.data['market_state'][-1] == 'BULLISH' or self.data['market_state'][-1] == 'BEARISH'
        c_er = self.data['ER'][-1] > self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_hour and candle_datetime.time().hour > self.start_trade_hour
        
        if c_er and c_market_structure and c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_trending(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [50]:
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

window = 2
global_tp_multiple = 3
global_stop_trade_hour = 20
global_start_trade_hour = 5
global_er15_threshold = 0.3

# start_date = '2023-3-20 00:00'
# end_date = '2023-7-7 00:00'
start_date = '2020-1-1 00:00'
end_date = '2026-12-30 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"


df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
# Affichage des transitions d'états
# df_result["state_changed"] = df_result["market_state"] != df_result[
#     "market_state"
# ].shift(1)
# print(df_result[df_result["state_changed"]][["Close", "market_state"]])
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    
    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        h1_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list
indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-01-15 14:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-02-26 08:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-01-01 22:00...
End                       2026-09-17 09:00...
Duration                   2450 days 10:00:00
Exposure Time [%]                     5.04587
Equity Final [$]                    11873.396
Equity Peak [$]                     15026.291
Return [%]                           18.73396
Buy & Hold Return [%]                44.93502
Return (Ann.) [%]                     2.08396
Volatility (Ann.) [%]                 6.44921
CAGR [%]                              2.59256
Sharpe Ratio                          0.32313
Sortino Ratio                         0.51672
Calmar Ratio                          0.09932
Alpha [%]                            16.11479
Beta                                  0.05829
Max. Drawdown [%]                   -20.98252
Avg. Drawdown [%]                    -1.48799
Max. Drawdown Duration      796 days 22:00:00
Avg. Drawdown Duration       34 days 13:00:00
# Trades                                  100
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p8356', ...)

In [37]:
import time
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

window = 2
global_tp_multiple = 6
global_stop_trade_hour = 21
global_start_trade_hour = 5
global_er15_threshold = 0.3

# start_date = '2023-3-20 00:00'
# end_date = '2023-7-7 00:00'
start_date = '2020-1-1 00:00'
end_date = '2022-12-30 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"


df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
# Affichage des transitions d'états
# df_result["state_changed"] = df_result["market_state"] != df_result[
#     "market_state"
# ].shift(1)
# print(df_result[df_result["state_changed"]][["Close", "market_state"]])
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    
    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        h1_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list
indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

start_hour_list = [4,5,6,7,8]
stop_hour_list = [16,17,18,19,20,21,22]
tp_multiple_list = [1,2,3,4,5,6,7,8,9,10]

with open(f"./results/result_{pair}_{time.time()}.csv", "a") as f:
    f.write(f"StartHour,StopHour,TpMultiple,Return,ReturnYearly,MDrawdown,WinRate,NumTrades\n")

    for start_hour in start_hour_list:
        for stop_hour in stop_hour_list:
            for tp_multiple in tp_multiple_list:
                global_start_trade_hour = start_hour
                global_stop_trade_hour = stop_hour
                global_tp_multiple = tp_multiple

                bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                stats = bt.run()
        
                f.write(f"{global_start_trade_hour},{global_stop_trade_hour},{global_tp_multiple},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']}, {stats['# Trades']}\n")
        
                print(f"start_hour = {start_hour} & stop_trade_hour = {stop_hour} & tp_multiple = {tp_multiple} DONE")

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 4 & stop_trade_hour = 16 & tp_multiple = 1 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-06-02 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-07-10 08:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 4 & stop_trade_hour = 16 & tp_multiple = 2 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-03-24 07:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-03-26 07:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

...

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-09-16 13:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-11-30 15:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 8 & stop_trade_hour = 22 & tp_multiple = 9 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-09-16 13:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-11-30 15:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

start_hour = 8 & stop_trade_hour = 22 & tp_multiple = 10 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-10-04 15:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-10-08 10:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http